In [1]:

import os
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"  # CUDA async allocator

import jax
from jax import config
config.update("jax_enable_x64", True)
from jax import numpy as jnp
import numpy as np

In [7]:
ng = 131
max_chol_size = 42
nc = -(-ng//max_chol_size)
print(nc)
# print()

4


In [8]:
print(ng/nc) 
ng_c = -(-ng//nc) 
# print(-(-Ng//Nc))
print(ng_c)
print(nc*ng_c)

32.75
33
132


In [11]:
print(-(-ng//ng_c))

4


In [6]:
import jax
stats = jax.devices()[0].memory_stats()
# print(stats["bytes_in_use"], stats["bytes_limit"])
print(stats)

None


In [2]:
def chunk_chol(chol, nchol_chunk = None, memory = None, spin_factor = 1):
    '''
    chunk the cholesky vectors (Ng, M, M) 
    to (Nc, Ng_max, M, M). The size of Ng
    is determined by allowed memory (in bytes).
    Nc * Ng_max not necessarily = Ng. The 
    Cholesky vectors maybe padded to have
    the same chunk size.

    Input
    chol: Cholesky Vectors (Ng, M, M)
    nchol_chunk: number of chol per chunk (use if memory is not set)
    memory: allowed memory per walker in MB
    spin_factor : 1 for restricted and generalized 2 for unrestricted
    
    Return
    chunked chol in shape (Nc, Ng_max, M, M)
    '''
    if (nchol_chunk is None) == (memory is None):
        raise ValueError("Specify exactly one of `nchol_chunk` or `memory`.")
    
    if not (isinstance(spin_factor, int) and spin_factor >= 1):
        raise ValueError(f"spin_factor must be a positive integer, got {spin_factor!r}")
    
    Ng, M1, M2 = chol.shape

    if memory is not None:
        bytes_per_element = 16  # complex128
        # max Ng per chunk that fits in the memory budget
        bytes_per_vector = M1 * M2 * bytes_per_element
        max_chunk_size = int(memory * 1024**2 // bytes_per_vector)
    else:
        max_chunk_size = nchol_chunk

    max_chunk_size = max_chunk_size // spin_factor

    if max_chunk_size < 1:
        raise ValueError(
            f"Chunk size after spin_factor={spin_factor} division is < 1. "
            f"Increase memory budget or nchol_chunk "
            f"(M1={M1}, M2={M2}, bytes_per_vector="
            f"{M1 * M2 * 16 / 1024**2:.3f} MB)."
        )

    # pad minimum number of zero-vectors to chol 
    # s.t. the padded chol / Nc <= max_chunk_size
    Nc = -(-Ng // max_chunk_size)   # ceil(Ng / max_chunk_size)
    Ng_max = -(-Ng // Nc)           # ceil(Ng / Nc)

    # pad the tail with zeros so every chunk has identical shape
    pad_amount = Nc * Ng_max - Ng
    if pad_amount > 0:
        chol = jnp.pad(chol, ((0, pad_amount), (0, 0), (0, 0)))

    return chol.reshape(Nc, Ng_max, M1, M2)

In [3]:
Ng, M1, M2 = 305, 50, 50   # 305 chosen to force non-trivial padding
key = jax.random.PRNGKey(0)
k_re, k_im = jax.random.split(key)
chol = (
    jax.random.normal(k_re, (Ng, M1, M2), dtype=jnp.float64)
    + 1j * jax.random.normal(k_im, (Ng, M1, M2), dtype=jnp.float64)
).astype(jnp.complex128)

print(f"Input shape: {chol.shape}, dtype: {chol.dtype}")
print(f"Input size:  {chol.nbytes / 1024**2:.2f} MB\n")

# ----- test 1: chunk by explicit count -----
print("=== Test 1: nchol_chunk=200, restricted ===")
out = chunk_chol(chol, nchol_chunk=200, spin_factor=1)
Nc, Ng_max, _, _ = out.shape
print(f"  output shape: {out.shape}")
print(f"  Nc={Nc}, Ng_max={Ng_max}, padding={Nc * Ng_max - Ng}")
# expected: Nc=2, Ng_max=ceil(305/2)=153, padding=1
assert (Nc, Ng_max) == (2, 153)
# real data preserved, padded tail is zero
assert jnp.allclose(out.reshape(-1, M1, M2)[:Ng], chol)
assert jnp.all(out.reshape(-1, M1, M2)[Ng:] == 0)

# ----- test 2: chunk by memory budget -----
print("\n=== Test 2: memory=50 MB, restricted ===")
# one vector = 50*50*16 B ≈ 0.0381 MB → ~1310 fit, but only 305 exist
out = chunk_chol(chol, memory=50, spin_factor=1)
Nc, Ng_max, _, _ = out.shape
print(f"  output shape: {out.shape}")
print(f"  Nc={Nc}, Ng_max={Ng_max}, padding={Nc * Ng_max - Ng}")
assert Nc == 1 and Ng_max == Ng   # no chunking needed

# ----- test 3: tight memory budget forces multiple chunks -----
print("\n=== Test 3: memory=4 MB, restricted ===")
# bytes_per_vector = 50*50*16 = 40000 B
# max_chunk_size = 4 MB / 40000 B ≈ 104
# Nc = ceil(305/104) = 3, Ng_max = ceil(305/3) = 102
out = chunk_chol(chol, memory=4, spin_factor=1)
Nc, Ng_max, _, _ = out.shape
print(f"  output shape: {out.shape}")
print(f"  Nc={Nc}, Ng_max={Ng_max}, padding={Nc * Ng_max - Ng}")
assert (Nc, Ng_max) == (3, 102)

# ----- test 4: unrestricted halves the chunk size -----
print("\n=== Test 4: nchol_chunk=200, unrestricted (spin_factor=2) ===")
out = chunk_chol(chol, nchol_chunk=200, spin_factor=2)
Nc, Ng_max, _, _ = out.shape
print(f"  output shape: {out.shape}")
print(f"  Nc={Nc}, Ng_max={Ng_max}, padding={Nc * Ng_max - Ng}")
# max_chunk_size = 200 // 2 = 100, Nc = ceil(305/100) = 4, Ng_max = ceil(305/4) = 77
assert (Nc, Ng_max) == (4, 77)

# ----- test 5: no padding needed (Ng divides evenly) -----
print("\n=== Test 5: Ng=300, nchol_chunk=150 (no padding) ===")
chol_300 = chol[:300]
out = chunk_chol(chol_300, nchol_chunk=150)
Nc, Ng_max, _, _ = out.shape
print(f"  output shape: {out.shape}")
print(f"  Nc={Nc}, Ng_max={Ng_max}, padding={Nc * Ng_max - 300}")
assert (Nc, Ng_max) == (2, 150)
assert jnp.allclose(out.reshape(-1, M1, M2), chol_300)

# ----- test 6: non-square trailing dims -----
print("\n=== Test 6: non-square (Ng, nelec, norb) ===")
nelec, norb = 10, 50
chol_ne = (
    jax.random.normal(k_re, (Ng, nelec, norb), dtype=jnp.float64)
    + 1j * jax.random.normal(k_im, (Ng, nelec, norb), dtype=jnp.float64)
).astype(jnp.complex128)
out = chunk_chol(chol_ne, nchol_chunk=100)
Nc, Ng_max, d1, d2 = out.shape
print(f"  output shape: {out.shape}")
assert (d1, d2) == (nelec, norb)
assert (Nc, Ng_max) == (4, 77)   # ceil(305/100)=4, ceil(305/4)=77

# ----- test 7: input validation -----
print("\n=== Test 7: input validation ===")
try:
    chunk_chol(chol)   # neither specified
except ValueError as e:
    print(f"  caught (neither set):  {e}")
try:
    chunk_chol(chol, nchol_chunk=100, memory=10)   # both specified
except ValueError as e:
    print(f"  caught (both set):     {e}")
try:
    chunk_chol(chol, nchol_chunk=1, spin_factor=2)   # chunk size 0
except ValueError as e:
    print(f"  caught (chunk size 0): {e}")
try:
    chunk_chol(chol, nchol_chunk=100, spin_factor=0)   # bad spin_factor
except ValueError as e:
    print(f"  caught (spin=0):       {e}")

print("\nAll tests passed.")

Input shape: (305, 50, 50), dtype: complex128
Input size:  11.63 MB

=== Test 1: nchol_chunk=200, restricted ===
  output shape: (2, 153, 50, 50)
  Nc=2, Ng_max=153, padding=1

=== Test 2: memory=50 MB, restricted ===
  output shape: (1, 305, 50, 50)
  Nc=1, Ng_max=305, padding=0

=== Test 3: memory=4 MB, restricted ===
  output shape: (3, 102, 50, 50)
  Nc=3, Ng_max=102, padding=1

=== Test 4: nchol_chunk=200, unrestricted (spin_factor=2) ===
  output shape: (4, 77, 50, 50)
  Nc=4, Ng_max=77, padding=3

=== Test 5: Ng=300, nchol_chunk=150 (no padding) ===
  output shape: (2, 150, 50, 50)
  Nc=2, Ng_max=150, padding=0

=== Test 6: non-square (Ng, nelec, norb) ===
  output shape: (4, 77, 10, 50)

=== Test 7: input validation ===
  caught (neither set):  Specify exactly one of `nchol_chunk` or `memory`.
  caught (both set):     Specify exactly one of `nchol_chunk` or `memory`.
  caught (chunk size 0): Chunk size after spin_factor=2 division is < 1. Increase memory budget or nchol_chunk (